[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/upm-classes/ai_medical_imaging_2026/blob/main/notebooks/student/NB05_augmentation_audit.ipynb)

# NB05 · Augmentation and the plausibility audit
**Artificial Intelligence in Medical Imaging** · Day 2 · Lab 6 (25 min), then the plausibility debate

| | |
|---|---|
| Runtime | CPU is enough (a few seconds per epoch) |
| Data | PneumoniaMNIST 28 px, CC BY 4.0 |
| Checkpoints | validation images are not augmented · both runs use the same seed · audit table has at least 5 rated transforms |
| Tiers | **Core** = TODO cells · **Plus** = class-weighted loss · **Challenge** = a transform for another modality |

**Learning goals.** (1) Build a training-only augmentation pipeline with `torchvision.transforms.v2`. (2) Compare training with and without it, fairly. (3) Judge each transform as a clinician *and* as an engineer: the **plausibility audit** is the point of this lab, and it feeds the debate.

**Two rules.** Never augment validation or test images. Split by patient first, augment afterwards, so that copies of one image never sit in two sets.

In [ ]:
%pip install -q "medmnist==3.0.2"
import os, random, time, math
from pathlib import Path
import numpy as np, matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

def set_seed(seed=42):
    random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed); torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)
for d in ("data", "outputs"): Path(d).mkdir(exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)

In [ ]:
def load_pneumonia():
    """PneumoniaMNIST (28 px, CC BY 4.0). Falls back to SYNTHETIC data if the download fails."""
    try:
        from medmnist import PneumoniaMNIST
        out = {}
        for s in ("train", "val", "test"):
            d = PneumoniaMNIST(split=s, download=True, root="data")
            out[s] = (d.imgs.reshape(len(d.imgs), 28, 28).astype(np.uint8), d.labels.reshape(-1).astype(np.int64))
        return out, True
    except Exception as e:
        print("!! Could not download PneumoniaMNIST:", repr(e))
        print("!! Using SYNTHETIC data so the code still runs. Numbers below are NOT about pneumonia.")
        rng = np.random.default_rng(0); out = {}
        for s, n in (("train", 2000), ("val", 300), ("test", 300)):
            y = (rng.random(n) < 0.7).astype(np.int64)
            x = rng.normal(100, 25, (n, 28, 28)); x[y == 1, 8:20, 8:20] += 40
            out[s] = (x.clip(0, 255).astype(np.uint8), y)
        return out, False

data, REAL = load_pneumonia()
to_tensor = lambda x: ((torch.from_numpy(x).float() / 255.0 - 0.5) / 0.5).unsqueeze(1)    # (N, 1, 28, 28), range -1..1
X = {s: to_tensor(v[0]) for s, v in data.items()}
Y = {s: torch.from_numpy(v[1]) for s, v in data.items()}
print({s: tuple(X[s].shape) for s in X}, "| real data:", REAL)
maj_val = max(Y["val"].float().mean().item(), 1 - Y["val"].float().mean().item())
print(f"majority-class rate on validation: {maj_val:.3f}  (an 'always the same answer' model scores this)")

In [ ]:
import torchvision
from torchvision.transforms import v2
print("torchvision", torchvision.__version__)

def build_cnn(filters=(8, 16), n_out=2):
    layers, c, size = [], 1, 28
    for f in filters:
        layers += [nn.Conv2d(c, f, 3), nn.ReLU(), nn.MaxPool2d(2)]; c, size = f, (size - 2) // 2
    return nn.Sequential(*layers, nn.Flatten(), nn.Linear(c * size * size, n_out))

class XrayDS(torch.utils.data.Dataset):
    """uint8 images (N, 28, 28) + labels + a transform applied on the fly."""
    def __init__(self, imgs, labels, transform): self.imgs, self.labels, self.transform = imgs, labels, transform
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        t = self.transform(self.imgs[i]).as_subclass(torch.Tensor)       # plain tensor (1, 28, 28)
        return t, int(self.labels[i])

def fit(model, train_ds, val_ds, epochs=8, lr=1e-3, bs=128, seed=SEED, class_weights=None, verbose=True):
    set_seed(seed); model.to(device)
    g = torch.Generator().manual_seed(seed)
    tr = DataLoader(train_ds, batch_size=bs, shuffle=True, generator=g)
    va = DataLoader(val_ds, batch_size=512, shuffle=False)
    w = None if class_weights is None else torch.tensor(class_weights, dtype=torch.float32, device=device)
    crit = nn.CrossEntropyLoss(weight=w); opt = torch.optim.Adam(model.parameters(), lr=lr)
    h = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "seed": seed}
    for ep in range(epochs):
        model.train(); tl = tc = n = 0
        for xb, yb in tr:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); out = model(xb); loss = crit(out, yb); loss.backward(); opt.step()
            tl += loss.item() * len(xb); tc += (out.argmax(1) == yb).sum().item(); n += len(xb)
        model.eval(); vl = vc = vn = 0
        with torch.no_grad():
            for xb, yb in va:
                xb, yb = xb.to(device), yb.to(device); out = model(xb)
                vl += crit(out, yb).item() * len(xb); vc += (out.argmax(1) == yb).sum().item(); vn += len(xb)
        h["train_loss"].append(tl / n); h["train_acc"].append(tc / n); h["val_loss"].append(vl / vn); h["val_acc"].append(vc / vn)
        if verbose: print(f"epoch {ep+1:2d} | train loss {tl/n:.3f} acc {tc/n:.3f} | val loss {vl/vn:.3f} acc {vc/vn:.3f}")
    return h
print("helpers ready")

## 1. Build the pipelines (8 min)
Two pipelines that share the **same preprocessing** (to tensor, scale to 0…1, normalise to −1…1):

- `eval_tf`: preprocessing only. Used for validation and test.
- `train_tf`: preprocessing **plus** random, mild augmentation. Used for training only.

We use *mild* settings on purpose: a rotation of ±8°, a shift of at most 5%, a brightness change of ±10%. Rotation fills the new corners with black (`fill=0`), a small artefact to remember.

**TODO 1.** Fill both pipelines. Hint: start with `v2.ToImage()` (it accepts the NumPy image) and end with `v2.ToDtype(torch.float32, scale=True)` then `v2.Normalize([0.5], [0.5])`.

In [ ]:
# TODO 1a: eval_tf = Compose([ToImage, ToDtype(float32, scale=True), Normalize([0.5],[0.5])])
eval_tf = None
# TODO 1b: train_tf = the same, with ColorJitter(brightness=0.1), RandomRotation(degrees=8)
#          and RandomAffine(degrees=0, translate=(0.05, 0.05)) inserted after ToImage()
train_tf = None
train_ds = XrayDS(data["train"][0], data["train"][1], train_tf)
# TODO 1c: the validation set must use the pipeline WITHOUT augmentation
val_ds = XrayDS(data["val"][0], data["val"][1], None)

### Checkpoint 1 · the validation set is not augmented

In [ ]:
names = [type(t).__name__ for t in val_ds.transform.transforms]
assert val_ds.transform is eval_tf, "validation must use eval_tf"
assert not any(n.startswith("Random") or n == "ColorJitter" for n in names), f"random transform found in the validation pipeline: {names}"
a, b = val_ds[0][0], val_ds[0][0]
assert torch.equal(a, b), "the same validation image must always give the same tensor"
assert a.shape == (1, 28, 28) and a.dtype == torch.float32 and -1.01 <= a.min() and a.max() <= 1.01
assert not torch.equal(train_ds[0][0], train_ds[0][0]), "training images should differ each time they are loaded"
print("Checkpoint 1 passed. Validation pipeline:", names)

In [ ]:
set_seed(SEED)
img = data["train"][0][0]
fig, ax = plt.subplots(2, 5, figsize=(11, 4.6)); axf = ax.ravel()
axf[0].imshow(img, cmap="gray", vmin=0, vmax=255); axf[0].set_title("original")
axf[4].imshow(eval_tf(img).as_subclass(torch.Tensor)[0], cmap="gray", vmin=-1, vmax=1); axf[4].set_title("eval_tf (no change)")
for k, i in enumerate([1, 2, 3, 5, 6, 7, 8, 9]):
    axf[i].imshow(train_tf(img).as_subclass(torch.Tensor)[0], cmap="gray", vmin=-1, vmax=1); axf[i].set_title(f"variant {k+1}")
for a in axf: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

**Look at the eight variants.** Which one would a radiologist still accept as a possible X-ray? Which looks suspicious? (Keep this in mind for the audit.)

## 2. Train with and without augmentation (9 min)
A fair comparison changes **one thing**: same model, same seed, same number of epochs, same validation set. Only the training pipeline differs.

In [ ]:
no_aug_ds = XrayDS(data["train"][0], data["train"][1], eval_tf)       # training set without augmentation
h_plain = fit(build_cnn(), no_aug_ds, val_ds, epochs=8, seed=SEED, verbose=False)
h_aug = fit(build_cnn(), train_ds, val_ds, epochs=8, seed=SEED, verbose=False)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
for name, h in [("no augmentation", h_plain), ("augmentation", h_aug)]:
    ax[0].plot(h["val_loss"], label=name + " (val)"); ax[0].plot(h["train_loss"], "--", label=name + " (train)")
    ax[1].plot(h["val_acc"], label=name + " (val)"); ax[1].plot(h["train_acc"], "--", label=name + " (train)")
ax[0].set_title("loss"); ax[1].set_title("accuracy"); ax[1].axhline(maj_val, color="grey", ls=":")
for a in ax: a.set_xlabel("epoch"); a.legend(fontsize=7)
plt.tight_layout(); plt.show()
for name, h in [("no augmentation", h_plain), ("augmentation", h_aug)]:
    print(f"{name:16s} final train acc {h['train_acc'][-1]:.3f} | final val acc {h['val_acc'][-1]:.3f} | gap {h['train_acc'][-1]-h['val_acc'][-1]:+.3f}")

In [ ]:
assert h_plain["seed"] == h_aug["seed"] == SEED, "both runs must use the same seed"
if REAL: assert h_aug["val_acc"][-1] > maj_val and h_plain["val_acc"][-1] > maj_val
print("Checkpoint 2 passed: same seed, both above the majority-class rate")

**Reflection.** (1) Did augmentation change the *training* accuracy? the *validation* accuracy? Is the validation difference larger than the ±1.6-point noise of 524 images? Re-run with `seed=1` before you believe a difference. (2) Training accuracy under augmentation is measured on augmented images. Is it comparable with the other run's? (3) Why is it wrong to report the augmented validation accuracy as the model's performance?

## 3. The plausibility audit (8 min)
For every transform ask two questions in one sentence each:
- **Clinical lens:** could this image exist in a real patient or scanner? could it change the diagnosis?
- **Engineering lens:** does it help generalisation? could the model learn a shortcut from it?

Rate each: **Y** plausible · **?** plausible with limits · **N** not plausible. One row is filled in as an example.

**TODO 2.** Rate **at least five more** transforms. Candidates: horizontal flip, vertical flip, rotation 90°, brightness ±10%, strong gamma change, strong Gaussian blur, random erasing (cutout), small crop and resize.

In [ ]:
import pandas as pd
audit = pd.DataFrame([
    ("Rotation 8 degrees", "Y", "Patients are rarely perfectly straight", "Black corners appear; keep angles small"),
    # TODO 2: add at least five more rows: (transform, "Y" / "?" / "N", one sentence clinical lens, one sentence engineering lens)

], columns=["transform", "plausible (Y/?/N)", "why (clinical lens)", "risk (engineering lens)"])
audit

### Checkpoint 3 · the audit table

In [ ]:
assert len(audit) >= 6, "the example row plus at least five of your own"
assert audit["plausible (Y/?/N)"].isin(["Y", "?", "N"]).all(), "ratings must be Y, ? or N"
assert (audit[["why (clinical lens)", "risk (engineering lens)"]].apply(lambda c: c.str.len() > 10)).all().all(), "one real sentence in each cell"
assert audit["plausible (Y/?/N)"].nunique() >= 2, "a table where everything has the same rating is not an audit: is nothing doubtful?"
audit.to_csv("outputs/plausibility_audit.csv", index=False)
print("Checkpoint 3 passed. Saved outputs/plausibility_audit.csv: bring it to the debate")

> **Misconception to avoid.** "A flip is always a free augmentation." It is free only when left and right are interchangeable for the task. In a chest X-ray the heart side and the L/R marker carry information.

> **Debate (next, 20 min).** Your group gets one transform: two of you defend it, two veto it, 90 seconds each. Use this table.

### Plus · class-weighted loss
In this training set pneumonia is the majority class. A weighted loss makes mistakes on the rarer class cost more. Compute weights inversely proportional to class frequency, train with `class_weights`, and compare **recall and specificity** (not just accuracy) with the unweighted run.

In [ ]:
# [PLUS]
# TODO (Plus): weights = n_samples / (2 * count of each class); train with class_weights=weights;
# compare recall (pneumonia) and specificity (normal) between the weighted and unweighted runs.
...

### Challenge · a transform for another modality
Pick a modality (CT, MRI, ultrasound, histology). Name **one augmentation you consider plausible** there and **one you would veto**, with a clinical and an engineering reason each (at least 40 characters per answer).

In [ ]:
# [CHALLENGE]
modality = ""
plausible = ""      # TODO (Challenge): one plausible augmentation + reason
veto = ""           # TODO (Challenge): one augmentation you would veto + reason
assert len(plausible) >= 40 and len(veto) >= 40, "write real justifications"

> **A caution about the data split.** PneumoniaMNIST's training and validation sets come from a 9:1 split of the source training set (MedMNIST v2 paper); the paper's description does not say the split is by patient. If images of the same child are in both sets, validation accuracy is optimistic. We return to this on Day 3 (leakage detective).

## Summary
- Augment the training set only; build one pipeline for training and a separate one without randomness for validation and test.
- Compare with and without under the same seed, and ask whether the gap exceeds the noise.
- An augmentation is a claim about what changes do **not** change the diagnosis. The audit makes that claim explicit.

*Next (Day 3):* NB06 (ROC, calibration, thresholds and a leakage demo).